<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/EmbeddingsAnalysis_TopicClustring_Topic2Vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Clustring

### Clustering Topic Models from TurfTopic

1. TurfTopic Default model and configuration
2. Use other models like ModernBert

In [ ]:
# Install libraries and packages
!pip install 'turftopic[umap-learn, datamapplot]'

In [2]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [3]:
# Read and Load dataset
dataset = pd.read_csv('sample_PubMedDataAbstracts.csv')

# Show the datasets
### Abstract Embeddings Sample Dataset
print('Node Content:', dataset.shape)
print(dataset)

Node Content: (10000, 4)
      Unnamed: 0                                              title  \
0              0  Phenotypic variability of Niemann-Pick disease...   
1              1  Recurrent hypoglycemia secondary to metformin ...   
2              2  Adaptation of the Ambulatory and Home Care Rec...   
3              3  Multidimensional family therapy in adolescents...   
4              4  Balanced crystalloids versus isotonic saline i...   
...          ...                                                ...   
9995        9995  Methylmercury in Industrial Harbor Sediments i...   
9996        9996  Factors Affecting Secondhand Smoke Avoidance B...   
9997        9997  Predicting Infectious Disease Using Deep Learn...   
9998        9998  Diosgenin Glucoside Protects against Spinal Co...   
9999        9999  Omics Approaches for Engineering Wheat Product...   

                                               abstract  year  
0     Background Niemann-Pick disease type C (NPC) i...  2

In [4]:
# Extract only the 'abstract' column and drop others
abstracts = dataset['abstract'].dropna().reset_index(drop=True)

# Display a few samples to verify
print(abstracts)

0       Background Niemann-Pick disease type C (NPC) i...
1       Background Metformin toxicity is well known to...
2       Background Measuring service use and costs is ...
3       Background Substance use and delinquency are c...
4       Objectives Intravenous fluids are one of the m...
                              ...                        
9995    The distribution of methylmercury (MeHg) and t...
9996    The purpose of this study was to examine the s...
9997    Infectious disease occurs when a person is inf...
9998    Spinal cord injury (SCI) is a severe traumatic...
9999    Abiotic stresses greatly influenced wheat prod...
Name: abstract, Length: 10000, dtype: object


### 1. TurfTopic Default model and configuration

In [5]:
# Import topic clustring required libraries
from sentence_transformers import SentenceTransformer
from turftopic import Top2Vec

# Import normalize
from sklearn.preprocessing import normalize

In [ ]:
# Using TurfTopic default encoder to extract embedding of the dataset
encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(abstracts, show_progress_bar=True)

In [7]:
# # Manually L2 normalize the embeddings
# embeddings = normalize(embeddings, axis=1)

In [8]:
# Show embeddings matrix and Check the dimention of each eambeding
print(embeddings,"\n\n", embeddings.shape)

[[-0.06247491 -0.06421211 -0.04776807 ...  0.05779283 -0.00046217
  -0.0348533 ]
 [ 0.03635849 -0.05012973 -0.0077231  ... -0.00230364  0.01381705
  -0.08019841]
 [ 0.01247236  0.01453279 -0.01938152 ...  0.00787823  0.01995303
  -0.02415804]
 ...
 [-0.00543451 -0.08347747  0.03953594 ... -0.01684228 -0.10415071
   0.11270402]
 [-0.06943101 -0.0625765  -0.00057785 ...  0.00205806  0.01154448
  -0.04431521]
 [-0.07437815  0.00144461 -0.0106479  ... -0.01505966 -0.03949417
  -0.08860712]] 

 (10000, 384)


In [ ]:
# umap_args={'n_neighbors': 15, 'min_dist': 0.1, 'low_memory': False}
# hdbscan_args={'min_cluster_size': 15, 'min_samples': 5}

# Training model (Uses HDBSCAN and umap)
model = Top2Vec(encoder=encoder, random_state=42)
topic_data = model.prepare_topic_data(abstracts, embeddings=embeddings)

In [10]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ metabolomic, metabolomics, biomarkers, proteomics, proteomic, pharmacologically, pharmacological,    │
│          │ biomarker, pharmacology, metabolome                                                                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ retinopathy, retinal, intraocular, glaucoma, cataract, ocular, retina, ophthalmic, corneal, macular  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ implant, implants, osteoclasts, osteoblasts, osteogenic, osteoclast, implantation, implanted, bone,  │
│          │ osteoblast                                                                                           │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ dental, tooth, oral, teeth, periodontitis, periodontal, dent, caries, orally, hygiene                │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ periodontitis, periodontal, gingival, dental, cytokine, oral, inflammation, cytokines, tooth,        │
│          │ interleukin                                                                                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ dental, teeth, tooth, periodontal, dent, enamel, periodontitis, maxillary, resin, oral               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ judgments, integrity, ethics, inherent, normative, judgment, moral, interpretation, interpretations, │
│          │ necessitates                                                                                         │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        6 │ pollutants, epidemiology, pollution, epidemiological, epidemiologic, pollutant, asthma, polluted,    │
│          │ toxicity, inhaled                                                                                    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        7 │ microscopy, microscope, fluorescence, imaging, immunofluorescence, subcellular, proteomics,          │
│          │ intracellular, intercellular, nanoscale                                                              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │ tuberculosis, mycobacterium, mycobacterial, tb, antibacterial, bactericidal, pathogen, bacteremia,   │
│          │ pneumococcal, pathogens                                                                              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        9 │ classifiers, classifying, classification, classify, supervised, classifier, cnn, bioinformatics,     │
│          │ biomarker, datasets                                                                                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │ ligands, ligand, compounds, synthesized, molecule, synthesis, catalysis, molecular, aromatic,        │
│          │ catalyzed                                  

In [11]:
# Cluster model hierarchy
model.hierarchy.cut(3).plot_tree()

In [12]:
# Merging topics to reduce number of topics
model.reduce_topics(n_reduce_to=25)
print(model.hierarchy.cut(3))

Root: 
├── -1: metabolomic, metabolomics, biomarkers, proteomics, proteomic, pharmacologically, 
│   pharmacological, biomarker, 
│   pharmacology, metabolome
├── 11: sensor, sensing, sensors, accelerometers, gps, accelerometer, iot, tracking, monitoring, 
│   802
├── 104: species, speciation, ecology, phenotypic, phylogenies, biodiversity, fauna, phylogenetic, 
│   taxonomic, 
│   ecological
│   ├── 42: pollination, flowering, plants, pollen, floral, phenotypic, botanical, speciation, flower,
│   │   cultivars
│   └── 43: species, ecology, speciation, fauna, phylogenies, biodiversity, phenotypic, phylogenetic,
│       taxonomic, ecological
├── 113: phytochemical, phytochemicals, flavonoids, antioxidants, antioxidant, antioxidative, 
│   metabolites, phenolics, 
│   flavonoid, herbal
│   ├── 52: phytochemicals, metabolites, antimicrobials, phytochemical, alkaloids, bioactivities, 
│   │   alkaloid, biosynthesis, 
│   │   biogeochemical, biochemical
│   └── 53: phytochemical, phytochemi

In [14]:
# Model hierarchy after merging topics
fig = model.hierarchy[156].plot_tree()
fig.show()

In [15]:
# We will reset the hierarchy, so that we can see all topics at once.
model.reset_topics()
fig = model.plot_clusters_datamapplot(hover_text=dataset["title"])
fig.show()

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_data.py:258: UserWarning:

Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_data.py:277: UserWarning:

Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 



### Calculate and display standard deviation


In [27]:
# Show embeddings matrix and Check the dimention of each eambeding
print(embeddings,"\n\n", embeddings.shape)

[[-0.06247491 -0.06421211 -0.04776807 ...  0.05779283 -0.00046217
  -0.0348533 ]
 [ 0.03635849 -0.05012973 -0.0077231  ... -0.00230364  0.01381705
  -0.08019841]
 [ 0.01247236  0.01453279 -0.01938152 ...  0.00787823  0.01995303
  -0.02415804]
 ...
 [-0.00543451 -0.08347747  0.03953594 ... -0.01684228 -0.10415071
   0.11270402]
 [-0.06943101 -0.0625765  -0.00057785 ...  0.00205806  0.01154448
  -0.04431521]
 [-0.07437815  0.00144461 -0.0106479  ... -0.01505966 -0.03949417
  -0.08860712]] 

 (10000, 384)


In [16]:
# Calculate the standard deviation across all elements in the embeddings matrix
overall_std_dev = np.std(embeddings)
print(f"Overall Standard Deviation of Embeddings: {overall_std_dev:.4f}")


Overall Standard Deviation of Embeddings: 0.0510


In [34]:
# Calculate the standard deviation for each dimension (column) of the embeddings
std_dev_per_dimension = np.std(embeddings, axis=0)
print("\nStandard Deviation per Embedding Dimension (first 10 dimensions):")
print(std_dev_per_dimension[:10]) # Print first 10 for brevity


Standard Deviation per Embedding Dimension (first 10 dimensions):
[0.04943018 0.05297992 0.04895246 0.05080314 0.05615833 0.0493866
 0.05057677 0.05560939 0.04570052 0.04802996]


In [18]:
# descriptive statistics (mean and min/max) for std dev per dimension
print(f"\nMinimum Standard Deviation across dimensions: {np.min(std_dev_per_dimension):.4f}")
print(f"Maximum Standard Deviation across dimensions: {np.max(std_dev_per_dimension):.4f}")
print(f"Average Standard Deviation across dimensions: {np.mean(std_dev_per_dimension):.4f}")


Minimum Standard Deviation across dimensions: 0.0000
Maximum Standard Deviation across dimensions: 0.0643
Average Standard Deviation across dimensions: 0.0489


In [44]:
# Define a small tolerance for floating-point comparison
# check if it's very close to 0. Standard deviation will almost never be exactly 0 due to floating point math.
tolerance = 1e-8

# Find the indices of dimensions where standard deviation is approximately zero
zero_std_dev_indices = np.where(std_dev_per_dimension < tolerance)[0]

if len(zero_std_dev_indices) > 0:
    print("Embedding dimensions with standard deviation approximately 0:")
    print(zero_std_dev_indices)
    print(f"Number of dimensions: {len(zero_std_dev_indices)}")
else:
    print("No embedding dimensions found with standard deviation approximately 0.")

# inspect the standard deviation values for these dimensions
print("\nStandard deviations with 0 value for these dimensions:")
print(std_dev_per_dimension[zero_std_dev_indices])

Embedding dimensions with standard deviation approximately 0:
[127 223 319]
Number of dimensions: 3

Standard deviations with 0 value for these dimensions:
[0.000000e+00 0.000000e+00 5.395018e-09]


In [73]:
# Find the indices of dimensions where standard deviation is approximately zero
zero_std_dev_indices = np.where(std_dev_per_dimension < tolerance)[0]

if len(zero_std_dev_indices) > 0:
    print("Embedding dimensions with standard deviation approximately 0:")
    print(zero_std_dev_indices)
    print(f"Number of such dimensions: {len(zero_std_dev_indices)}")
    print("\n--- Exact elements in these dimensions across all rows ---")
    # Iterate through each identified dimension
    for dim_idx in zero_std_dev_indices:
        print(f"\nDimension Index: {dim_idx} (Standard Deviation: {std_dev_per_dimension[dim_idx]:.10f})")
        # Extract the entire column for this dimension
        elements_in_this_dimension = embeddings[:, dim_idx]
        print(f"Elements in Dimension {dim_idx} (first 10 and last 10, if many):")

        # If there are many elements, print a sample, otherwise print all
        if len(elements_in_this_dimension) > 20:
            print(elements_in_this_dimension[:384])
            # print("...")
            # print(elements_in_this_dimension[-10:])
        else:
            print(elements_in_this_dimension)
        print("-" * 50) # Separator for clarity
else:
    print("No embedding dimensions found with standard deviation approximately 0.")


Embedding dimensions with standard deviation approximately 0:
[127 223 319]
Number of such dimensions: 3

--- Exact elements in these dimensions across all rows ---

Dimension Index: 127 (Standard Deviation: 0.0000000000)
Elements in Dimension 127 (first 10 and last 10, if many):
[ 6.14512925e-33  4.80709048e-33  3.96042267e-33  5.83392630e-33
  3.28252353e-33  1.75171415e-33  2.36736977e-33  1.30512833e-33
  4.28340701e-33  4.96210732e-33  3.39182981e-33  1.25919285e-33
  3.82145463e-33  3.18606466e-33  4.89248720e-33  3.84813541e-33
 -3.71547936e-34  2.69799004e-33  2.24708529e-33  5.70722601e-33
  6.80620826e-34  3.15466207e-33  1.87632225e-33  3.19067517e-33
  8.32620722e-33  3.17499114e-33  1.42642125e-33  3.43605888e-33
  2.99417182e-33  2.41351527e-33  2.09418176e-33  4.03462869e-33
  7.39999040e-33  1.39787097e-33  3.63032365e-33  1.93520955e-33
  4.20550773e-33  4.22961235e-33  4.89151154e-33  3.43975030e-33
  5.70286346e-33  5.32251830e-33  5.06656653e-33  4.96294707e-33
  2.